# Forecasting Hands and Objects, joined to narrations

This notebook loads the Ego4D Forecasting Hands and Objects train and validation tables and joins each verb to one narration sentence. Every count, filter, and figure comes from `src/eg04d_join.py`. The cells below only call that module.

The official files are not stored in this repository. Accept the license at [ego4d-data.org](https://ego4d-data.org/), then download them into `data/ego4d`:

```bash
python -m ego4d.cli.cli --output_directory data/ego4d --datasets annotations --benchmarks FHO
```

The download must produce `fho_lta_train.json`, `fho_lta_val.json`, `narrations.json`, and `ego4d.json`. `build_joined_table(..., download=True)` runs that same command when the files are missing. It does not bypass the license.

## Setup

Import the join module and point it at the local annotation directory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "eg04d_join.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import eg04d_join as join

DATA_DIR = ROOT / "data" / "ego4d"
DATA_DIR

## Load the joined table

Repeated forecast rows collapse to one action. A narration is kept only when its timestamp falls inside the action window. The revised narration pass is used when a video has one. The camera-wearer tag is removed, and the verb, including the words inside a label such as `take (pick, grab, get)`, is removed from the sentence.

In [ ]:
paths = join.resolve_annotation_paths(DATA_DIR)
notes, counts = join.build_joined_table(DATA_DIR)
narrations = join.load_narrations(paths.narrations)
scoring = join.scoring_rows(notes)
notes.shape

## Descriptive statistics

Joined rows still include ambiguous matches. Scoring rows are the ones a later model is allowed to see.

In [ ]:
join.descriptive_statistics(notes)

## Verb frequency

The distribution below is the scoring set: one row per collapsed action that joined a single narration.

In [ ]:
join.verb_frequency(scoring)

In [ ]:
join.plot_verb_frequency(scoring)

## Narrations per video

These counts come from the narration file, before the join. Each video contributes the revised pass when that pass exists, and the original pass otherwise.

In [ ]:
per_video = join.narrations_per_video(narrations)
join.narration_video_summary(per_video)

In [ ]:
join.plot_narrations_per_video(per_video)

## Preprocessing

The join already removed a leading `C` or `#C` tag and deleted the labeled verb from each sentence. `verb_remains` is true when a form of that verb is still in the leftover, which means the strip missed it. Ambiguous rows, where two sentences inside the window are about equally close, stay in the table and are dropped by `scoring_rows`.

In [ ]:
join.preprocessing_summary(notes)

In [ ]:
join.preprocessing_examples(notes)

## Preliminary results

Collapsed actions are the denominator. Rows surviving the join matched one narration and were not ambiguous. Ambiguous rows are excluded from scoring. The last count is validation rows whose participant also appears on a training action. The split is by video, so the same person can sit on both sides.

In [ ]:
join.counts_table(counts)